# 🧬 Chromatin Factor Paralog Synthetic Lethality Scanner
## DepMap Public 25Q3 - Real Data Analysis

---

**作業ディレクトリ**: `/content/drive/MyDrive/2026/SR_paralog`

### 概要
DepMap Public 25Q3 実データを用いた、クロマチンリモデリング因子パラログの合成致死性一括スキャン

### 特徴
- **SWI/SNF, PRC, NuRD, INO80, ISWI** 等のクロマチン複合体パラログを網羅（60ペア）
- **statsmodels OLS** による交絡補正（組織、発現量、コピー数）
- **多重検定補正**（FDR, Benjamini-Hochberg法）

---
## 1. 環境セットアップ

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

# 作業ディレクトリ設定
WORK_DIR = '/content/drive/MyDrive/2026/SR_paralog'
DATA_DIR = f'{WORK_DIR}/depmap_25q3'

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

print(f"Work directory: {WORK_DIR}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
# パッケージインストール
!pip install statsmodels -q

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import false_discovery_control
import statsmodels.api as sm
import statsmodels.formula.api as smf
from typing import List, Dict, Tuple, Optional, Set
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# プロット設定
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

print("✅ Setup complete!")

---
## 2. DepMap 25Q3 データのダウンロード

DepMap Portalから直接ダウンロードします。

**注意**: ファイルサイズが大きいため、初回は時間がかかります（合計約1GB）

In [ ]:
# DepMap 25Q3 ダウンロードURL
# 最新のURLは https://depmap.org/portal/download/all/ で確認してください

DEPMAP_FILES = {
    'CRISPRGeneEffect.csv': 'https://figshare.com/ndownloader/files/XXXXXXX',  # 要更新
    'Model.csv': 'https://figshare.com/ndownloader/files/XXXXXXX',
    'OmicsSomaticMutations.csv': 'https://figshare.com/ndownloader/files/XXXXXXX',
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv': 'https://figshare.com/ndownloader/files/XXXXXXX',
    'OmicsCNGene.csv': 'https://figshare.com/ndownloader/files/XXXXXXX',
}

print("""⚠️  DepMap 25Q3 データのダウンロード手順:

方法1: 手動ダウンロード（推奨）
1. https://depmap.org/portal/download/all/ にアクセス
2. 'DepMap Public 25Q3' を選択
3. 以下のファイルをダウンロード:
   - CRISPRGeneEffect.csv (~200MB)
   - Model.csv (~5MB)
   - OmicsSomaticMutations.csv (~500MB)
   - OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv (~100MB)
   - OmicsCNGene.csv (~50MB)
4. Google Drive の以下のフォルダにアップロード:
   {}

方法2: wgetでダウンロード（URLが判明している場合）
次のセルのURLを更新して実行
""".format(DATA_DIR))

In [ ]:
# 方法2: wgetでダウンロード（URLを更新してから実行）
# コメントを外して実行

# !wget -O "{DATA_DIR}/CRISPRGeneEffect.csv" "YOUR_URL_HERE"
# !wget -O "{DATA_DIR}/Model.csv" "YOUR_URL_HERE"
# !wget -O "{DATA_DIR}/OmicsSomaticMutations.csv" "YOUR_URL_HERE"
# !wget -O "{DATA_DIR}/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv" "YOUR_URL_HERE"
# !wget -O "{DATA_DIR}/OmicsCNGene.csv" "YOUR_URL_HERE"

In [ ]:
# データファイルの存在確認
required_files = [
    'CRISPRGeneEffect.csv',
    'Model.csv',
    'OmicsSomaticMutations.csv',
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',
]

optional_files = ['OmicsCNGene.csv']

print("📁 Checking data files...\n")

all_required_exist = True
for f in required_files:
    path = f"{DATA_DIR}/{f}"
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    size = f"({os.path.getsize(path) / 1e6:.1f} MB)" if exists else "(MISSING)"
    print(f"  {status} {f} {size}")
    if not exists:
        all_required_exist = False

print("\nOptional files:")
for f in optional_files:
    path = f"{DATA_DIR}/{f}"
    exists = os.path.exists(path)
    status = "✅" if exists else "⚠️"
    size = f"({os.path.getsize(path) / 1e6:.1f} MB)" if exists else "(not found)"
    print(f"  {status} {f} {size}")

if all_required_exist:
    print("\n✅ All required files found! Ready to proceed.")
else:
    print("\n❌ Missing required files. Please download from DepMap Portal.")

---
## 3. クロマチン因子パラログデータベース

In [ ]:
@dataclass
class ChromatinParalogPair:
    """クロマチン因子パラログペア"""
    gene_a: str           # 変異/低発現遺伝子（ドライバー）
    gene_b: str           # 依存性ターゲット（パラログ）
    complex_name: str     # 所属複合体
    subunit_type: str     # サブユニットタイプ
    evidence_level: str   # エビデンスレベル
    pmid: str = ""        # 参考文献PMID


# クロマチンリモデリング複合体のパラログペア
CHROMATIN_PARALOG_DATABASE = [
    # ========== SWI/SNF (BAF/PBAF) Complex ==========
    ChromatinParalogPair("SMARCA4", "SMARCA2", "SWI/SNF", "ATPase", "published", "26552009"),
    ChromatinParalogPair("SMARCA2", "SMARCA4", "SWI/SNF", "ATPase", "published", "26552009"),
    ChromatinParalogPair("ARID1A", "ARID1B", "SWI/SNF (BAF)", "DNA-binding", "published", "24520176"),
    ChromatinParalogPair("ARID1B", "ARID1A", "SWI/SNF (BAF)", "DNA-binding", "published", "24520176"),
    ChromatinParalogPair("ARID2", "ARID1A", "SWI/SNF (PBAF)", "DNA-binding", "predicted", ""),
    ChromatinParalogPair("BCL7A", "BCL7B", "SWI/SNF", "Accessory", "predicted", ""),
    ChromatinParalogPair("BCL7B", "BCL7C", "SWI/SNF", "Accessory", "predicted", ""),
    ChromatinParalogPair("BRD9", "BRD7", "SWI/SNF (ncBAF/PBAF)", "Bromodomain", "published", "31253568"),
    ChromatinParalogPair("BRD7", "BRD9", "SWI/SNF (PBAF/ncBAF)", "Bromodomain", "published", "31253568"),
    ChromatinParalogPair("DPF1", "DPF2", "SWI/SNF (BAF)", "PHD-finger", "predicted", ""),
    ChromatinParalogPair("DPF2", "DPF3", "SWI/SNF (BAF)", "PHD-finger", "predicted", ""),
    ChromatinParalogPair("SMARCC1", "SMARCC2", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("SMARCC2", "SMARCC1", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("SMARCD1", "SMARCD2", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("SMARCD2", "SMARCD3", "SWI/SNF", "Core", "predicted", ""),
    ChromatinParalogPair("PBRM1", "ARID2", "SWI/SNF (PBAF)", "Scaffold", "published", "29562155"),
    
    # ========== Polycomb Repressive Complex (PRC) ==========
    ChromatinParalogPair("CBX2", "CBX4", "PRC1", "Chromodomain", "predicted", ""),
    ChromatinParalogPair("CBX4", "CBX8", "PRC1", "Chromodomain", "predicted", ""),
    ChromatinParalogPair("CBX7", "CBX8", "PRC1", "Chromodomain", "predicted", ""),
    ChromatinParalogPair("PCGF1", "PCGF2", "PRC1", "RING-finger", "predicted", ""),
    ChromatinParalogPair("PCGF2", "PCGF4", "PRC1 (cPRC1)", "RING-finger", "predicted", ""),
    ChromatinParalogPair("PCGF4", "PCGF5", "PRC1", "RING-finger", "predicted", ""),
    ChromatinParalogPair("RING1", "RNF2", "PRC1", "E3-ligase", "predicted", ""),
    ChromatinParalogPair("RNF2", "RING1", "PRC1", "E3-ligase", "predicted", ""),
    ChromatinParalogPair("EZH1", "EZH2", "PRC2", "Methyltransferase", "published", "31068699"),
    ChromatinParalogPair("EZH2", "EZH1", "PRC2", "Methyltransferase", "published", "31068699"),
    ChromatinParalogPair("SUZ12", "EED", "PRC2", "Core", "predicted", ""),
    
    # ========== NuRD Complex ==========
    ChromatinParalogPair("CHD3", "CHD4", "NuRD", "ATPase", "predicted", ""),
    ChromatinParalogPair("CHD4", "CHD5", "NuRD", "ATPase", "predicted", ""),
    ChromatinParalogPair("MTA1", "MTA2", "NuRD", "Scaffold", "predicted", ""),
    ChromatinParalogPair("MTA2", "MTA3", "NuRD", "Scaffold", "predicted", ""),
    ChromatinParalogPair("HDAC1", "HDAC2", "NuRD/Sin3", "Deacetylase", "published", ""),
    ChromatinParalogPair("HDAC2", "HDAC1", "NuRD/Sin3", "Deacetylase", "published", ""),
    ChromatinParalogPair("GATAD2A", "GATAD2B", "NuRD", "Accessory", "predicted", ""),
    ChromatinParalogPair("MBD2", "MBD3", "NuRD", "MBD", "predicted", ""),
    
    # ========== INO80 Complex ==========
    ChromatinParalogPair("INO80", "SRCAP", "INO80/SRCAP", "ATPase", "predicted", ""),
    ChromatinParalogPair("ACTR5", "ACTR8", "INO80", "Actin-related", "predicted", ""),
    
    # ========== ISWI Complex ==========
    ChromatinParalogPair("SMARCA1", "SMARCA5", "ISWI", "ATPase", "predicted", ""),
    ChromatinParalogPair("SMARCA5", "SMARCA1", "ISWI", "ATPase", "predicted", ""),
    ChromatinParalogPair("BAZ1A", "BAZ1B", "ISWI (ACF/WICH)", "Bromodomain", "predicted", ""),
    ChromatinParalogPair("BAZ2A", "BAZ2B", "ISWI (NoRC)", "Bromodomain", "predicted", ""),
    
    # ========== Histone Acetyltransferases (HAT) ==========
    ChromatinParalogPair("CREBBP", "EP300", "HAT", "Acetyltransferase", "published", "27571770"),
    ChromatinParalogPair("EP300", "CREBBP", "HAT", "Acetyltransferase", "published", "27571770"),
    ChromatinParalogPair("KAT2A", "KAT2B", "SAGA/ATAC", "Acetyltransferase", "predicted", ""),
    ChromatinParalogPair("KAT6A", "KAT6B", "HBO1/MOZ", "Acetyltransferase", "predicted", ""),
    
    # ========== Histone Methyltransferases ==========
    ChromatinParalogPair("KMT2A", "KMT2B", "COMPASS", "H3K4me", "predicted", ""),
    ChromatinParalogPair("KMT2C", "KMT2D", "COMPASS", "H3K4me", "published", ""),
    ChromatinParalogPair("SETD1A", "SETD1B", "COMPASS", "H3K4me", "predicted", ""),
    ChromatinParalogPair("NSD1", "NSD2", "H3K36me", "Methyltransferase", "predicted", ""),
    ChromatinParalogPair("NSD2", "NSD3", "H3K36me", "Methyltransferase", "predicted", ""),
    
    # ========== Histone Demethylases ==========
    ChromatinParalogPair("KDM1A", "KDM1B", "LSD", "H3K4/K9 demethylase", "predicted", ""),
    ChromatinParalogPair("KDM4A", "KDM4B", "JMJD2", "H3K9/K36 demethylase", "predicted", ""),
    ChromatinParalogPair("KDM5A", "KDM5B", "JARID1", "H3K4 demethylase", "predicted", ""),
    ChromatinParalogPair("KDM6A", "KDM6B", "UTX/JMJD3", "H3K27 demethylase", "published", ""),
    
    # ========== Cohesin Complex ==========
    ChromatinParalogPair("STAG1", "STAG2", "Cohesin", "SA subunit", "published", "33007256"),
    ChromatinParalogPair("STAG2", "STAG1", "Cohesin", "SA subunit", "published", "33007256"),
    
    # ========== DNA Methylation ==========
    ChromatinParalogPair("DNMT1", "DNMT3A", "DNA methylation", "Methyltransferase", "predicted", ""),
    ChromatinParalogPair("DNMT3A", "DNMT3B", "DNA methylation", "Methyltransferase", "predicted", ""),
    ChromatinParalogPair("TET1", "TET2", "DNA demethylation", "Dioxygenase", "predicted", ""),
    ChromatinParalogPair("TET2", "TET3", "DNA demethylation", "Dioxygenase", "predicted", ""),
]

print(f"Total paralog pairs in database: {len(CHROMATIN_PARALOG_DATABASE)}")

# 複合体ごとの集計
complex_counts = pd.Series([p.complex_name for p in CHROMATIN_PARALOG_DATABASE]).value_counts()
print("\nPairs by complex:")
display(complex_counts.to_frame('count').head(15))

---
## 4. データローダー

In [ ]:
class DepMap25Q3Loader:
    """DepMap Public 25Q3 データローダー"""
    
    FILE_MAPPING = {
        'crispr': 'CRISPRGeneEffect.csv',
        'model': 'Model.csv', 
        'mutations': 'OmicsSomaticMutations.csv',
        'expression': 'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',
        'cnv': 'OmicsCNGene.csv',
    }
    
    def __init__(self, data_dir: str):
        self.data_dir = Path(data_dir)
        self.crispr: Optional[pd.DataFrame] = None
        self.model: Optional[pd.DataFrame] = None
        self.mutations: Optional[pd.DataFrame] = None
        self.expression: Optional[pd.DataFrame] = None
        self.cnv: Optional[pd.DataFrame] = None
        
    def load_all(self) -> 'DepMap25Q3Loader':
        """全データの一括読み込み"""
        print("="*60)
        print("Loading DepMap 25Q3 data...")
        print("="*60)
        self.load_crispr()
        self.load_model()
        self.load_mutations()
        self.load_expression()
        self.load_cnv()
        self._align_samples()
        print("="*60)
        return self
    
    def load_crispr(self) -> pd.DataFrame:
        """CRISPR依存性スコア"""
        filepath = self.data_dir / self.FILE_MAPPING['crispr']
        print(f"\n📊 Loading CRISPR data...")
        self.crispr = pd.read_csv(filepath, index_col=0)
        # カラム名正規化: "TP53 (7157)" → "TP53"
        self.crispr.columns = [c.split(' ')[0] for c in self.crispr.columns]
        print(f"   ✅ {self.crispr.shape[0]} cell lines × {self.crispr.shape[1]} genes")
        return self.crispr
    
    def load_model(self) -> pd.DataFrame:
        """細胞株メタデータ"""
        filepath = self.data_dir / self.FILE_MAPPING['model']
        print(f"\n📋 Loading Model data...")
        self.model = pd.read_csv(filepath)
        self.model = self.model.set_index('ModelID')
        print(f"   ✅ {len(self.model)} cell lines")
        
        # 組織分布
        if 'OncotreeLineage' in self.model.columns:
            print("\n   Top lineages:")
            for lin, count in self.model['OncotreeLineage'].value_counts().head(5).items():
                print(f"     - {lin}: {count}")
        return self.model
    
    def load_mutations(self) -> pd.DataFrame:
        """変異データ"""
        filepath = self.data_dir / self.FILE_MAPPING['mutations']
        print(f"\n🧬 Loading Mutation data...")
        self.mutations = pd.read_csv(filepath, low_memory=False)
        print(f"   ✅ {len(self.mutations)} mutations")
        print(f"   Unique genes: {self.mutations['HugoSymbol'].nunique()}")
        return self.mutations
    
    def load_expression(self) -> pd.DataFrame:
        """発現データ"""
        filepath = self.data_dir / self.FILE_MAPPING['expression']
        print(f"\n📈 Loading Expression data...")
        self.expression = pd.read_csv(filepath, index_col=0)
        self.expression.columns = [c.split(' ')[0] for c in self.expression.columns]
        print(f"   ✅ {self.expression.shape[0]} cell lines × {self.expression.shape[1]} genes")
        return self.expression
    
    def load_cnv(self) -> pd.DataFrame:
        """コピー数データ"""
        filepath = self.data_dir / self.FILE_MAPPING['cnv']
        if not filepath.exists():
            print(f"\n⚠️  CNV data not found, skipping...")
            return None
        print(f"\n📉 Loading CNV data...")
        self.cnv = pd.read_csv(filepath, index_col=0)
        self.cnv.columns = [c.split(' ')[0] for c in self.cnv.columns]
        print(f"   ✅ {self.cnv.shape[0]} cell lines × {self.cnv.shape[1]} genes")
        return self.cnv
    
    def _align_samples(self):
        """サンプルIDの整合性確認"""
        print(f"\n🔗 Aligning samples...")
        crispr_samples = set(self.crispr.index)
        model_samples = set(self.model.index)
        common = crispr_samples.intersection(model_samples)
        print(f"   Common samples (CRISPR ∩ Model): {len(common)}")
        
        if self.expression is not None:
            expr_common = common.intersection(set(self.expression.index))
            print(f"   With expression data: {len(expr_common)}")

---
## 5. 交絡補正付きOLS解析クラス

In [ ]:
class ConfounderAdjustedAnalyzer:
    """statsmodels OLS を用いた交絡補正付き解析"""
    
    def __init__(self, loader: DepMap25Q3Loader):
        self.loader = loader
        
    def get_lof_status(self, gene: str) -> pd.Series:
        """
        Loss-of-Function 変異ステータスを取得
        """
        if self.loader.mutations is None:
            return pd.Series(dtype=bool)
        
        # Damaging変異のタイプ
        damaging_types = {
            'Nonsense_Mutation', 'Frame_Shift_Del', 'Frame_Shift_Ins',
            'Splice_Site', 'Nonstop_Mutation', 'Start_Codon_Del',
            'De_novo_Start_OutOfFrame', 'Start_Codon_SNP'
        }
        
        # LikelyLoF または isDeleterious カラムがあれば使用
        mask = (self.loader.mutations['HugoSymbol'] == gene)
        
        if 'VariantType' in self.loader.mutations.columns:
            mask &= self.loader.mutations['VariantType'].isin(damaging_types)
        
        if 'LikelyLoF' in self.loader.mutations.columns:
            mask |= (
                (self.loader.mutations['HugoSymbol'] == gene) &
                (self.loader.mutations['LikelyLoF'] == True)
            )
        
        if 'isDeleterious' in self.loader.mutations.columns:
            mask |= (
                (self.loader.mutations['HugoSymbol'] == gene) &
                (self.loader.mutations['isDeleterious'] == True)
            )
        
        gene_muts = self.loader.mutations[mask]
        lof_lines = set(gene_muts['ModelID'].unique())
        all_lines = self.loader.crispr.index
        
        return pd.Series(
            [line in lof_lines for line in all_lines],
            index=all_lines,
            name=f'{gene}_LOF'
        )
    
    def get_low_expression_status(self, gene: str, percentile: float = 25) -> pd.Series:
        """低発現ステータスを取得"""
        if self.loader.expression is None or gene not in self.loader.expression.columns:
            return pd.Series(dtype=bool)
        
        expr = self.loader.expression[gene]
        threshold = np.nanpercentile(expr, percentile)
        return pd.Series(expr <= threshold, name=f'{gene}_low_expr')
    
    def get_cnv_status(self, gene: str, cn_threshold: float = 0.7) -> pd.Series:
        """コピー数欠失ステータス"""
        if self.loader.cnv is None or gene not in self.loader.cnv.columns:
            return pd.Series(dtype=bool)
        
        cnv = self.loader.cnv[gene]
        return pd.Series(cnv < -cn_threshold, name=f'{gene}_del')
    
    def build_regression_data(self, 
                               driver_gene: str,
                               target_gene: str,
                               stratify_by: str = "mutation") -> pd.DataFrame:
        """
        回帰分析用のデータフレームを構築
        """
        if target_gene not in self.loader.crispr.columns:
            return pd.DataFrame()
        
        # 基本データ
        data = pd.DataFrame({'dependency': self.loader.crispr[target_gene]})
        
        # ドライバーステータス
        if stratify_by == "mutation":
            data['driver_status'] = self.get_lof_status(driver_gene)
        elif stratify_by == "expression":
            data['driver_status'] = self.get_low_expression_status(driver_gene)
        elif stratify_by == "cnv":
            data['driver_status'] = self.get_cnv_status(driver_gene)
        elif stratify_by == "any":
            lof = self.get_lof_status(driver_gene)
            low_expr = self.get_low_expression_status(driver_gene)
            cnv_del = self.get_cnv_status(driver_gene)
            data['driver_status'] = lof | low_expr | cnv_del
        
        # 組織タイプ（交絡因子）
        if self.loader.model is not None and 'OncotreeLineage' in self.loader.model.columns:
            lineage = self.loader.model['OncotreeLineage'].reindex(data.index)
            top_lineages = lineage.value_counts().head(10).index.tolist()
            for lin in top_lineages:
                data[f'lineage_{lin}'] = (lineage == lin).astype(int)
        
        # ターゲット遺伝子の発現量（交絡因子）
        if self.loader.expression is not None and target_gene in self.loader.expression.columns:
            data['target_expr'] = self.loader.expression[target_gene].reindex(data.index)
        
        # ターゲット遺伝子のCNV（交絡因子）
        if self.loader.cnv is not None and target_gene in self.loader.cnv.columns:
            data['target_cnv'] = self.loader.cnv[target_gene].reindex(data.index)
        
        return data.dropna()
    
    def run_ols_analysis(self,
                          driver_gene: str,
                          target_gene: str,
                          stratify_by: str = "mutation",
                          min_affected: int = 5) -> Optional[Dict]:
        """
        OLS回帰による交絡補正付き解析
        """
        data = self.build_regression_data(driver_gene, target_gene, stratify_by)
        
        if len(data) < 20:
            return None
        
        n_affected = data['driver_status'].sum()
        n_unaffected = len(data) - n_affected
        
        if n_affected < min_affected or n_unaffected < min_affected:
            return None
        
        # 回帰式構築
        covariates = [col for col in data.columns 
                      if col.startswith('lineage_') or col in ['target_expr', 'target_cnv']]
        
        formula = 'dependency ~ driver_status'
        if covariates:
            formula += ' + ' + ' + '.join(covariates)
        
        try:
            model = smf.ols(formula, data=data).fit()
            
            coef_name = 'driver_status[T.True]' if 'driver_status[T.True]' in model.params else 'driver_status'
            coef = model.params.get(coef_name, np.nan)
            pvalue = model.pvalues.get(coef_name, np.nan)
            se = model.bse.get(coef_name, np.nan)
            
            ci = model.conf_int()
            ci_lower = ci.loc[coef_name, 0] if coef_name in ci.index else np.nan
            ci_upper = ci.loc[coef_name, 1] if coef_name in ci.index else np.nan
            
            affected_mean = data[data['driver_status'] == True]['dependency'].mean()
            unaffected_mean = data[data['driver_status'] == False]['dependency'].mean()
            
            return {
                'driver_gene': driver_gene,
                'target_gene': target_gene,
                'stratify_by': stratify_by,
                'n_affected': int(n_affected),
                'n_unaffected': int(n_unaffected),
                'n_total': len(data),
                'affected_mean': affected_mean,
                'unaffected_mean': unaffected_mean,
                'raw_delta': affected_mean - unaffected_mean,
                'adjusted_coef': coef,
                'adjusted_se': se,
                'adjusted_pvalue': pvalue,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'standardized_coef': coef / data['dependency'].std(),
                'r_squared': model.rsquared,
                'n_covariates': len(covariates),
            }
        except Exception as e:
            return None

---
## 6. バッチスキャナー

In [ ]:
class ChromatinParalogScanner:
    """クロマチン因子パラログの一括スキャン"""
    
    def __init__(self, loader: DepMap25Q3Loader):
        self.loader = loader
        self.analyzer = ConfounderAdjustedAnalyzer(loader)
        self.results: pd.DataFrame = pd.DataFrame()
        
    def scan_all_pairs(self,
                       paralog_pairs: List[ChromatinParalogPair] = None,
                       stratify_by: str = "any",
                       min_affected: int = 5,
                       verbose: bool = True) -> pd.DataFrame:
        """
        全パラログペアの一括スキャン
        """
        if paralog_pairs is None:
            paralog_pairs = CHROMATIN_PARALOG_DATABASE
        
        print(f"\n{'='*60}")
        print(f"Scanning {len(paralog_pairs)} chromatin paralog pairs...")
        print(f"Stratification: {stratify_by}")
        print(f"Min affected samples: {min_affected}")
        print(f"{'='*60}")
        
        results = []
        
        for i, pair in enumerate(paralog_pairs):
            if verbose and (i + 1) % 10 == 0:
                print(f"  Progress: {i+1}/{len(paralog_pairs)}")
            
            result = self.analyzer.run_ols_analysis(
                driver_gene=pair.gene_a,
                target_gene=pair.gene_b,
                stratify_by=stratify_by,
                min_affected=min_affected
            )
            
            if result:
                result['complex'] = pair.complex_name
                result['subunit_type'] = pair.subunit_type
                result['evidence_level'] = pair.evidence_level
                result['pmid'] = pair.pmid
                results.append(result)
        
        df = pd.DataFrame(results)
        
        if len(df) > 0:
            df['fdr'] = false_discovery_control(df['adjusted_pvalue'], method='bh')
            df = df.sort_values('adjusted_pvalue')
            df['significant_fdr05'] = df['fdr'] < 0.05
            df['significant_fdr10'] = df['fdr'] < 0.10
        
        self.results = df
        print(f"\n✅ Scan complete! {len(df)} pairs analyzed.")
        return df
    
    def get_summary_by_complex(self) -> pd.DataFrame:
        """複合体ごとの結果サマリー"""
        if len(self.results) == 0:
            return pd.DataFrame()
        
        return self.results.groupby('complex').agg({
            'driver_gene': 'count',
            'significant_fdr05': 'sum',
            'adjusted_coef': 'mean',
            'adjusted_pvalue': 'min',
        }).rename(columns={
            'driver_gene': 'n_pairs_tested',
            'significant_fdr05': 'n_significant',
            'adjusted_coef': 'mean_effect',
            'adjusted_pvalue': 'best_pvalue',
        }).sort_values('n_significant', ascending=False)

---
## 7. 可視化関数

In [ ]:
def plot_volcano(results: pd.DataFrame, title: str = "Chromatin Paralog Synthetic Lethality",
                 save_path: str = None):
    """Volcano plot"""
    fig, ax = plt.subplots(figsize=(12, 8))
    
    df = results.copy()
    df['neg_log_p'] = -np.log10(df['adjusted_pvalue'])
    
    colors = ['red' if (fdr < 0.05 and coef < -0.1) else 
              'orange' if (fdr < 0.1 and coef < -0.1) else 'lightgray'
              for fdr, coef in zip(df['fdr'], df['adjusted_coef'])]
    
    ax.scatter(df['adjusted_coef'], df['neg_log_p'], c=colors, alpha=0.6, s=60,
               edgecolors='black', linewidths=0.5)
    
    # ラベル
    for _, row in df[df['fdr'] < 0.1].nsmallest(15, 'adjusted_pvalue').iterrows():
        ax.annotate(f"{row['driver_gene']}→{row['target_gene']}",
                   (row['adjusted_coef'], row['neg_log_p']),
                   fontsize=8, fontweight='bold' if row['fdr'] < 0.05 else 'normal')
    
    ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5, label='p=0.05')
    ax.axvline(-0.1, color='green', linestyle='--', alpha=0.5, label='coef=-0.1')
    
    ax.set_xlabel('Adjusted Coefficient', fontsize=12)
    ax.set_ylabel('-log10(p-value)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend()
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_forest(results: pd.DataFrame, n_top: int = 20, save_path: str = None):
    """Forest plot"""
    df = results.nsmallest(n_top, 'adjusted_pvalue').sort_values('adjusted_coef')
    
    fig, ax = plt.subplots(figsize=(10, max(6, n_top * 0.4)))
    y_pos = np.arange(len(df))
    
    xerr = np.array([df['adjusted_coef'] - df['ci_lower'],
                    df['ci_upper'] - df['adjusted_coef']])
    
    colors = ['red' if fdr < 0.05 else 'orange' if fdr < 0.1 else 'gray' for fdr in df['fdr']]
    
    ax.errorbar(df['adjusted_coef'], y_pos, xerr=xerr, fmt='o', color='black',
                ecolor='gray', capsize=3, markersize=8)
    ax.scatter(df['adjusted_coef'], y_pos, c=colors, s=100, zorder=5)
    ax.axvline(0, color='black', linestyle='-', alpha=0.3)
    
    labels = [f"{r['driver_gene']}→{r['target_gene']} ({r['complex']})" for _, r in df.iterrows()]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Adjusted Coefficient (95% CI)', fontsize=12)
    ax.set_title(f'Top {n_top} Hits', fontsize=14)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_heatmap(results: pd.DataFrame, save_path: str = None):
    """Heatmap"""
    pivot = results.pivot_table(values='adjusted_coef', index='driver_gene',
                                columns='target_gene', aggfunc='mean')
    
    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns)*0.8), 
                                    max(8, len(pivot)*0.5)))
    sns.heatmap(pivot, cmap='RdBu_r', center=0, annot=True, fmt='.2f', ax=ax)
    ax.set_title('Paralog Dependencies (Adjusted Coefficient)', fontsize=14)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_single_pair(driver: str, target: str, loader: DepMap25Q3Loader, save_path: str = None):
    """個別ペアの詳細プロット"""
    analyzer = ConfounderAdjustedAnalyzer(loader)
    data = analyzer.build_regression_data(driver, target, stratify_by="any")
    
    if len(data) == 0:
        print(f"No data for {driver} - {target}")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    plot_data = pd.DataFrame({
        'Dependency': data['dependency'],
        'Group': data['driver_status'].map({True: f'{driver} LOF', False: f'{driver} WT'})
    })
    
    # Violin
    sns.violinplot(data=plot_data, x='Group', y='Dependency', ax=axes[0])
    sns.stripplot(data=plot_data, x='Group', y='Dependency', color='black', alpha=0.3, size=3, ax=axes[0])
    axes[0].axhline(-1, color='red', linestyle='--', alpha=0.5)
    axes[0].set_title(f'{target} Dependency')
    
    # Box
    sns.boxplot(data=plot_data, x='Group', y='Dependency', ax=axes[1])
    axes[1].axhline(-1, color='red', linestyle='--', alpha=0.5)
    axes[1].set_title(f'{target} Dependency Distribution')
    
    plt.suptitle(f'{driver} → {target} Synthetic Lethality', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()
    
    # 統計
    affected = data[data['driver_status'] == True]['dependency']
    unaffected = data[data['driver_status'] == False]['dependency']
    t_stat, p_val = stats.ttest_ind(affected, unaffected)
    
    print(f"\n{driver} LOF: n={len(affected)}, mean={affected.mean():.3f}")
    print(f"{driver} WT:  n={len(unaffected)}, mean={unaffected.mean():.3f}")
    print(f"Delta: {affected.mean() - unaffected.mean():.3f}")
    print(f"t-test p-value: {p_val:.2e}")

---
## 8. 🚀 解析実行

### 8.1 データ読み込み

In [ ]:
# データ読み込み
loader = DepMap25Q3Loader(data_dir=DATA_DIR)
loader.load_all()

### 8.2 一括スキャン実行

In [ ]:
# スキャナー初期化
scanner = ChromatinParalogScanner(loader)

# 一括スキャン実行
# stratify_by: "mutation", "expression", "cnv", "any"（OR条件）
results = scanner.scan_all_pairs(
    stratify_by="any",  # 変異 OR 低発現 OR CNV欠失
    min_affected=5,
    verbose=True
)

print(f"\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total pairs tested: {len(results)}")
print(f"Significant (FDR < 0.05): {results['significant_fdr05'].sum()}")
print(f"Significant (FDR < 0.10): {results['significant_fdr10'].sum()}")

### 8.3 結果の確認

In [ ]:
# Top 20 hits
print("\n" + "="*70)
print("TOP 20 HITS (by adjusted p-value)")
print("="*70)

display_cols = ['driver_gene', 'target_gene', 'complex', 'n_affected', 
                'raw_delta', 'adjusted_coef', 'adjusted_pvalue', 'fdr', 'evidence_level']

display(results.head(20)[display_cols])

In [ ]:
# 複合体別サマリー
print("\n" + "="*70)
print("SUMMARY BY COMPLEX")
print("="*70)

complex_summary = scanner.get_summary_by_complex()
display(complex_summary)

### 8.4 可視化

In [ ]:
# Volcano plot
plot_volcano(results, 
             title="Chromatin Paralog Synthetic Lethality (DepMap 25Q3)\nOLS Confounder Adjusted",
             save_path=f"{WORK_DIR}/results/volcano_plot.png")

In [ ]:
# Forest plot
plot_forest(results, n_top=20,
            save_path=f"{WORK_DIR}/results/forest_plot.png")

In [ ]:
# SWI/SNF heatmap
swisnf = results[results['complex'].str.contains('SWI/SNF', na=False)]
if len(swisnf) > 3:
    plot_heatmap(swisnf, save_path=f"{WORK_DIR}/results/swisnf_heatmap.png")

In [ ]:
# Top hitの詳細プロット
if len(results) > 0:
    top_hit = results.iloc[0]
    plot_single_pair(top_hit['driver_gene'], top_hit['target_gene'], loader,
                     save_path=f"{WORK_DIR}/results/top_hit_detail.png")

### 8.5 結果の保存

In [ ]:
# 結果をCSVに保存
results.to_csv(f"{WORK_DIR}/results/chromatin_paralog_scan_results.csv", index=False)
complex_summary.to_csv(f"{WORK_DIR}/results/chromatin_paralog_complex_summary.csv")

# 有意なhitsのみ
sig_results = results[results['fdr'] < 0.1]
sig_results.to_csv(f"{WORK_DIR}/results/significant_hits_fdr10.csv", index=False)

print(f"\n✅ Results saved to {WORK_DIR}/results/")
print(f"   - chromatin_paralog_scan_results.csv (all {len(results)} pairs)")
print(f"   - chromatin_paralog_complex_summary.csv")
print(f"   - significant_hits_fdr10.csv ({len(sig_results)} hits)")
print(f"   - volcano_plot.png")
print(f"   - forest_plot.png")
print(f"   - swisnf_heatmap.png")

---
## 9. 追加解析

### 9.1 特定のペアを詳細解析

In [ ]:
# 例: SMARCA4 → SMARCA2
plot_single_pair('SMARCA4', 'SMARCA2', loader,
                 save_path=f"{WORK_DIR}/results/SMARCA4_SMARCA2_detail.png")

In [ ]:
# 例: ARID1A → ARID1B
plot_single_pair('ARID1A', 'ARID1B', loader,
                 save_path=f"{WORK_DIR}/results/ARID1A_ARID1B_detail.png")

### 9.2 組織特異的解析

In [ ]:
# 肺癌のみで解析
def scan_tissue_specific(loader, tissue: str):
    """組織特異的スキャン"""
    # 組織でフィルタ
    tissue_lines = loader.model[loader.model['OncotreeLineage'] == tissue].index
    
    # フィルタしたデータで新しいローダーを作成
    tissue_loader = DepMap25Q3Loader(DATA_DIR)
    tissue_loader.crispr = loader.crispr.loc[loader.crispr.index.isin(tissue_lines)]
    tissue_loader.model = loader.model.loc[tissue_lines]
    tissue_loader.expression = loader.expression.loc[loader.expression.index.isin(tissue_lines)] if loader.expression is not None else None
    tissue_loader.cnv = loader.cnv.loc[loader.cnv.index.isin(tissue_lines)] if loader.cnv is not None else None
    tissue_loader.mutations = loader.mutations[loader.mutations['ModelID'].isin(tissue_lines)]
    
    print(f"\n{tissue} cell lines: {len(tissue_loader.crispr)}")
    
    scanner = ChromatinParalogScanner(tissue_loader)
    return scanner.scan_all_pairs(stratify_by="any", min_affected=3)

# 使用例（必要に応じてコメントを外す）
# lung_results = scan_tissue_specific(loader, 'Lung')
# lung_results.head(10)

### 9.3 カスタムパラログペアの追加

In [ ]:
# カスタムペアを定義して解析
custom_pairs = [
    # 追加したいペアをここに記述
    # ChromatinParalogPair("GENE_A", "GENE_B", "Complex", "Type", "user_defined"),
]

if custom_pairs:
    custom_scanner = ChromatinParalogScanner(loader)
    custom_results = custom_scanner.scan_all_pairs(custom_pairs, stratify_by="any")
    display(custom_results)

---
## 10. まとめ

### 解析パイプライン
```
DepMap 25Q3 Data
    ↓
Driver Status (LOF mutation / Low expression / CNV deletion)
    ↓
OLS Regression (dependency ~ driver + lineage + expression + CNV)
    ↓
FDR Correction
    ↓
Prioritized Hits
```

### 次のステップ
1. 有意なヒットの文献検証
2. 組織特異的解析（肺癌、大腸癌など）
3. 実験的バリデーション（CRISPR）
4. ドラッグターゲット評価